# Finetune TinyLlama to Generate Would You Rather Questions
Fine-tunes TinyLlama on a dataset of WYR questions so it can generate new ones in the same vibe.


In [1]:
import random
import json
from tqdm import tqdm

import os
from dataclasses import dataclass, field
from typing import Dict, List, Any

import json
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
import torch

/home/milk/Desktop/RESEARCH/wyr/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# Check if CUDA is available
print(f"Is CUDA available? {torch.cuda.is_available()}")

# Get the name of the GPU
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")


Is CUDA available? True
GPU Device: NVIDIA GeForce RTX 5060


In [3]:
# Load model directly
BASE_MODEL_ID = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    torch_dtype="auto"
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 470.31it/s]


In [4]:
# set padding token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

## Dataset
`formatted_polls.json` contains 313 WYR questions with `title`, `optionA`, and `optionB`.

Each example is formatted as a simple generation prompt so the model learns the style and structure of WYR questions:
```
Would you rather [optionA] or [optionB]?
```
At inference time, priming the model with `Would you rather` will generate new questions in the same vibe.


In [5]:
from datetime import datetime
import os

date = datetime.now().strftime("%Y-%m-%d")

POLLS_JSON    = "formatted_polls.json"   # <-- put your file here
JSONL_OUTPUT  = f"train-data/wyr_training_data-[{date}].jsonl"

os.makedirs("train-data", exist_ok=True)


In [6]:


def build_wyr_dataset(polls_path: str) -> Dataset:
    """
    Format each WYR poll as a plain generation string.
    The model learns to produce well-formed WYR questions by
    predicting the full text token-by-token.

    Two formats are included so the model learns both the
    full question and the two-option layout:

      Format B (structured):
        Would you rather...
        A) [optionA]
        B) [optionB]
    """
    with open(polls_path, 'r') as f:
        polls = json.load(f)

    rows = []
    for poll in polls:
        a = poll['optionA'].strip().rstrip('.')
        b = poll['optionB'].strip().rstrip('.')

        # Randomly swap A/B so the model doesn't develop order bias
        if random.random() < 0.5:
            a, b = b, a

        # Format B — structured two-option layout
        rows.append({'text': f"Would you rather...\nA) {a}\nB) {b}"})

    random.shuffle(rows)

    # Save for inspection
    with open(JSONL_OUTPUT, 'w') as f:
        for row in rows:
            f.write(json.dumps(row) + '\n')

    print(f"Built {len(rows)} training examples ({len(polls)} polls x 2 formats) -> {JSONL_OUTPUT}")
    return Dataset.from_list(rows)

train_dataset = build_wyr_dataset(POLLS_JSON)
print(train_dataset)
print("\nSample entries:")
for i in range(4):
    print(f"\n--- {i} ---")
    print(train_dataset[i]['text'])


Built 313 training examples (313 polls x 2 formats) -> train-data/wyr_training_data-[2026-05-13].jsonl
Dataset({
    features: ['text'],
    num_rows: 313
})

Sample entries:

--- 0 ---
Would you rather...
A) Morph into a fly when you get nervous (for 3 days)
B) Have inverted arms (thumb on wrong side)

--- 1 ---
Would you rather...
A) Sleep in wet clothes
B) Step in a puddle with socks on, and have to walk around like that for the whole day

--- 2 ---
Would you rather...
A) Look like any character you want to
B) Get 10 million USD

--- 3 ---
Would you rather...
A) Always have a charlie horse
B) Always have a migraine


## Train the model


In [7]:
# os.environ["OMP_NUM_THREADS"] = "6"
# os.environ["OPENBLAS_NUM_THREADS"] = "6"
# os.environ["MKL_NUM_THREADS"] = "6"
# os.environ["VECLIB_NUM_THREADS"] = "6"  
# os.environ["NUMEXPR_NUM_THREADS"] = "6"

# torch.set_num_threads(6)
# torch.set_num_interop_threads(2)

In [8]:
# Output directories
LORA_OUTPUT_DIR   = "./llama-wyr-lora"
MERGED_OUTPUT_DIR = "./llama-wyr-merged"

MAX_SEQ_LEN = 128  # WYR questions are short


peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

sft_config = SFTConfig(
    output_dir=LORA_OUTPUT_DIR,
    num_train_epochs=10,          # small dataset — multiple passes help
    dataloader_num_workers=4,
    dataloader_pin_memory=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=True,
    bf16=(
        torch.cuda.is_available()
        and torch.cuda.get_device_capability(0)[0] >= 8
    ),
    fp16=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.train()


[RANK 0] Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
[RANK 0] You are using packing, but the attention implementation is not set to a supported flash attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernel

Step,Training Loss
50,1.849604


TrainOutput(global_step=50, training_loss=1.849604034423828, metrics={'train_runtime': 39.3818, 'train_samples_per_second': 18.029, 'train_steps_per_second': 1.27, 'total_flos': 550052671979520.0, 'train_loss': 1.849604034423828})

In [9]:
# 7) Save LoRA adapter + tokenizer locally
os.makedirs(LORA_OUTPUT_DIR, exist_ok=True)
trainer.model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)
print(f"Saved LoRA adapter to {LORA_OUTPUT_DIR}")

# 8) Merge LoRA weights into base model and save a standalone model
print("Merging LoRA adapter into base model...")

# Reload base model on CPU (or cuda if you want)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="cpu",   # merge on CPU to avoid GPU OOM
)

lora_model = PeftModel.from_pretrained(base_model, LORA_OUTPUT_DIR)
merged_model = lora_model.merge_and_unload()  # apply LoRA weights into base

os.makedirs(MERGED_OUTPUT_DIR, exist_ok=True)
merged_model.save_pretrained(MERGED_OUTPUT_DIR)
tokenizer.save_pretrained(MERGED_OUTPUT_DIR)
print(f"Saved merged full model to {MERGED_OUTPUT_DIR}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Saved LoRA adapter to ./llama-wyr-lora
Merging LoRA adapter into base model...


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.07s/it]

Saved merged full model to ./llama-wyr-merged


## Generate new Would You Rather questions


In [10]:
# Quick generation test — prime the model with the WYR opener
# and let it complete the question.

model.eval()

prompts = [
    "Would you rather...",
    #"Would you rather...",
]
prompts *= 10

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.9,
            top_p=0.95,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"PROMPT: {prompt!r}")
    print(f"OUTPUT: {generated}")
    print()


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: 'Would you rather...'
OUTPUT: Would you rather...
A) Wish that your teeth were permanently glowing yellow/orange
B) Want all of your food to taste like french onion soup



[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: 'Would you rather...'
OUTPUT: Would you rather...
A) Get 10 bananas everyday for the rest of your life (not enough to eat!)
B) Eat one chicken leg per day



[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

PROMPT: 'Would you rather...'
OUTPUT: Would you rather...
A) Always be the last person to get your order
B) Be the first person to get their food



[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: 'Would you rather...'
OUTPUT: Would you rather...
A) A $1000 prize if you survive 2 hours in a room of spiders (but only the ones on bottom row will be deadly)



[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: 'Would you rather...'
OUTPUT: Would you rather...
A) Wizarding worlds, but only ever go to the 2nd level (spoiler: hp is more dangerous then harry potter universe)

PROMPT: 'Would you rather...'
OUTPUT: Would you rather...
A) Have unlimited wealth but no morals
B) Live in constant poverty with a minimal to zero morals

